# Task 1 — The carry-forward audit (15 claims)

Fifteen claims about **this dataset**, one from (almost) every week of the
course. For each: the week, true or false, and **the number that settles it**.
A verdict with no computed number earns nothing — so every verdict below is
computed from `data/` in the cells, not typed in.

Run all cells top to bottom from the `Task6/` folder.


In [1]:
import importlib.util
from pathlib import Path
import pandas as pd

NB_DIR = Path.cwd()
DATA = NB_DIR / "data"
print("notebook dir:", NB_DIR)
print("data dir exists:", DATA.exists())

# The pipeline script's helpers, loaded from the file sitting next to this
# notebook — the same strict rules as W6_su92-bsdsm-f23-018.py: plain-decimal
# numbers only, DD/MM/YYYY-or-ISO dates, positional extract (headers differ
# across the three files, so columns are read by position, not by name).
_spec = importlib.util.spec_from_file_location("w6pipe", NB_DIR / "W6_su92-bsdsm-f23-018.py")
w6 = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(w6)
CONTRACT, PLAIN, STATUS_MAP = w6.CONTRACT, w6.PLAIN, w6.STATUS_MAP
norm_id, plain_number, parse_date = w6.norm_id, w6.plain_number, w6.parse_date
extract_csv, extract_rail = w6.extract_csv, w6.extract_rail
find_source_files, mode_key, bizkey = w6.find_source_files, w6.mode_key, w6.bizkey
print("pipeline helpers loaded from W6_su92-bsdsm-f23-018.py")

notebook dir: /home/hatch/workspace/data-warehouse/week6/Task6
data dir exists: True


pipeline helpers loaded from W6_su92-bsdsm-f23-018.py


In [2]:
from collections import Counter

marine_path, rail_path, road_path = find_source_files(DATA / "march")
staging, artifacts = [], []
for path, kind in ((marine_path, "csv"), (rail_path, "xlsx"), (road_path, "csv")):
    rows, arts = extract_csv(path, path.name) if kind == "csv" else extract_rail(path, path.name)
    staging.extend(rows)
    artifacts.extend(arts)
    print(f"  {path.name}: {len(rows)} rows")
for a in artifacts:
    print(f"  extract artifact (NOT a row): {a[0]} line {a[1]} — {a[2]}")
stg = pd.DataFrame(staging)[CONTRACT + ["SourceFile", "SourceLine", "SourceRow"]]
print("per-file rows:", dict(Counter(stg["SourceFile"])))
print("staging rows:", len(stg))
assert len(stg) == 1284, "staging must be 1,284 rows"


  marine_march.csv: 261 rows
  rail_march.xlsx: 315 rows


  road_march.csv: 708 rows
  extract artifact (NOT a row): road_march.csv line 710 — blank line
  extract artifact (NOT a row): road_march.csv line 711 — TOTAL footer
per-file rows: {'marine_march.csv': 261, 'rail_march.xlsx': 315, 'road_march.csv': 708}
staging rows: 1284


In [3]:
alias = {r["RawSpelling"].strip(): int(r["CityKey"]) for r in
         pd.read_csv(DATA / "alias_lookup.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_city = pd.read_csv(DATA / "dims" / "DimCity.csv", dtype=str, keep_default_na=False)
city_name = {int(r["CityKey"]): r["CityName"] for r in dim_city.to_dict("records")}
route = {(int(r["OriginCityKey"]), int(r["DestCityKey"])): int(r["RouteKey"]) for r in
         pd.read_csv(DATA / "dims" / "DimRoute.csv", dtype=str, keep_default_na=False).to_dict("records")}
dim_status = pd.read_csv(DATA / "dims" / "DimStatus.csv", dtype=str, keep_default_na=False)
status_names = {int(r["StatusKey"]): r["StatusName"] for r in dim_status.to_dict("records")}
print(f"alias spellings: {len(alias)}; cities: {len(city_name)}; routes: {len(route)}; statuses: {status_names}")

alias spellings: 23; cities: 15; routes: 21; statuses: {1: 'Delivered', 2: 'In Transit', 3: 'Delayed', -1: 'Unknown'}


In [4]:
# ---- REJECT: the four classes (disjoint for March; verified below) ----
wtext = stg["WeightKG"].map(lambda v: plain_number(v) is None and (v or "").strip() != "")
baddate = stg["ShipDate"].map(lambda v: parse_date(v) is None)
wneg = stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0)
blankid = stg["ShipmentID"].map(lambda v: norm_id(v) == "")
assert not (wtext & baddate).any() and not (wtext & wneg).any() and not (wtext & blankid).any()
assert not (baddate & wneg).any() and not (baddate & blankid).any() and not (wneg & blankid).any()
rejected = stg[wtext | baddate | wneg | blankid].copy()
loadable = stg[~(wtext | baddate | wneg | blankid)].copy()
print(f"rejected: {len(rejected)} (weight-text {int(wtext.sum())}, date-unparseable {int(baddate.sum())}, "
      f"weight-negative {int(wneg.sum())}, id-blank {int(blankid.sum())}); loadable: {len(loadable)}")

# ---- DEDUP: TRIM + UPPER ShipmentID, first occurrence kept ----
loadable["_n"] = loadable["ShipmentID"].map(norm_id)
seen_raw, byte_removed = set(), 0
for _, row in loadable.iterrows():
    k = tuple(row[c] for c in CONTRACT)
    if k in seen_raw:
        byte_removed += 1
    else:
        seen_raw.add(k)
deduped = loadable.drop_duplicates(subset=["_n"], keep="first").copy()
removed = len(loadable) - len(deduped)
print(f"duplicates removed: {removed} ({byte_removed} byte-identical + {removed - byte_removed} trailing-space-only)")
deduped = deduped.drop(columns=["_n"])

# ---- TRANSFORM + ROUTE ----
work = deduped.copy()
is_rail = work["SourceFile"].map(lambda f: "rail" in f.lower())
wnum = work["WeightKG"].map(plain_number)
wkg = wnum.copy()
wkg[is_rail] = wkg[is_rail] * 1000
work["WeightKG_num"] = wkg.map(lambda v: int(round(v)))
work["FreightCostPKR_num"] = work["FreightCostPKR"].map(lambda v: int(round(plain_number(v))))
work["ShipmentID"] = work.apply(
    lambda r: ("RL-" + norm_id(r["ShipmentID"])
               if "rail" in r["SourceFile"].lower() and not norm_id(r["ShipmentID"]).startswith("RL-")
               else norm_id(r["ShipmentID"])), axis=1)
work["ShipDate_parsed"] = work["ShipDate"].map(parse_date)
work["DateKey"] = work["ShipDate_parsed"].map(lambda d: int(d.strftime("%Y%m%d")))
work["StatusKey"] = work["StatusRaw"].map(lambda v: STATUS_MAP.get((v or "").strip().upper(), ("", -1))[1])
work["OriginCityKey"] = work["Origin"].map(lambda v: alias.get((v or "").strip(), -1))
work["DestCityKey"] = work["Destination"].map(lambda v: alias.get((v or "").strip(), -1))
work["RouteKey"] = [route.get((o, d), -1) for o, d in zip(work["OriginCityKey"], work["DestCityKey"])]
work["ModeKey"] = work["SourceFile"].map(mode_key)

# ---- CONFORM CUSTOMERS: merge registry duplicates, keep the losing spelling as an alias ----
cust = pd.read_csv(DATA / "customers_master.csv", dtype=str, keep_default_na=False)
cust["num"] = cust["CustomerID"].map(lambda v: int(v.split("-")[1]))
cust["biz"] = cust["CustomerName"].map(bizkey)
groups = {}
for _, r in cust.iterrows():
    groups.setdefault(r["biz"], []).append(r)
alias_map, dim_rows = {}, []
for biz, members in groups.items():
    members = sorted(members, key=lambda r: r["num"])
    surv = members[0]
    for m in members[1:]:
        alias_map[m["num"]] = surv["num"]
    dim_rows.append({"CustomerKey": int(surv["num"]), "CustomerName": surv["CustomerName"],
                     "City": surv["City"], "Sector": surv["Sector"],
                     "Aliases": "; ".join(m["CustomerName"] for m in members[1:])})
work["CustomerKey"] = work["CustomerID"].map(
    lambda v: alias_map.get(int(v.split("-")[1]), int(v.split("-")[1])))
dim_customer = pd.DataFrame(dim_rows).sort_values("CustomerKey").reset_index(drop=True)
dim_customer = pd.concat([dim_customer, pd.DataFrame(
    [{"CustomerKey": -1, "CustomerName": "Unknown", "City": "Unknown", "Sector": "Unknown", "Aliases": ""}])],
    ignore_index=True)
print(f"registry rows: {len(cust)}; businesses after conform: {len(dim_customer) - 1}; merged pairs: {len(alias_map)}")

# ---- FACT (in memory; the pipeline script writes this to CSV) ----
fact = pd.DataFrame({
    "ShipmentID": work["ShipmentID"],
    "DateKey": work["DateKey"].astype(int),
    "OriginCityKey": work["OriginCityKey"].astype(int),
    "DestCityKey": work["DestCityKey"].astype(int),
    "RouteKey": work["RouteKey"].astype(int),
    "ModeKey": work["ModeKey"].astype(int),
    "StatusKey": work["StatusKey"].astype(int),
    "CustomerKey": work["CustomerKey"].astype(int),
    "WeightKG": work["WeightKG_num"].astype(int),
    "FreightCostPKR": work["FreightCostPKR_num"].astype(int),
    "ShipmentCount": 1})
print(f"fact rows: {len(fact)}; total weight: {int(fact['WeightKG'].sum()):,} kg; "
      f"total cost: PKR {int(fact['FreightCostPKR'].sum()):,}")

rejected: 24 (weight-text 9, date-unparseable 6, weight-negative 5, id-blank 4); loadable: 1260
duplicates removed: 60 (48 byte-identical + 12 trailing-space-only)
registry rows: 46; businesses after conform: 40; merged pairs: 6
fact rows: 1200; total weight: 17,542,050 kg; total cost: PKR 147,037,000


### The 15 claims — one computed verdict each

In [5]:
# Claims 1–5 — each verdict is a comparison on computed numbers.
# Claim 1 (Week 5): "The March files contain more rows than PakFreight shipped consignments in March."
v1 = len(stg) > len(fact)
print(f"1. Week 5 — staging {len(stg)} rows vs fact {len(fact)} rows -> {'TRUE' if v1 else 'FALSE'}")
print(f"   settling number: {len(stg)} vs {len(fact)} — staging holds every extracted row; only {len(fact)} load")
print(f"   (24 rejects + 60 duplicates = {24 + 60} = the {len(stg) - len(fact)}-row gap)")

# Claim 2 (Week 5): "Every negative number in the three files is a data error."
neg_w = int(stg["WeightKG"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0).sum())
neg_c = int(stg["FreightCostPKR"].map(lambda v: plain_number(v) is not None and plain_number(v) < 0).sum())
v2 = neg_c == 0
print(f"2. Week 5 — negatives: {neg_w} weights + {neg_c} costs = {neg_w + neg_c} in total -> {'TRUE' if v2 else 'FALSE'}")
print(f"   settling number: {neg_c} — the {neg_w} negative weights are errors, but the {neg_c} negative costs are real credit notes, loaded as negatives")

# Claim 3 (Week 2): "Each team can total its own file and the three totals will add up to the warehouse total."
def file_sum(fname, col):
    sub = stg[stg["SourceFile"] == fname]
    return sum(plain_number(v) for v in sub[col] if plain_number(v) is not None)
mar_w, rail_t, road_w = file_sum("marine_march.csv", "WeightKG"), file_sum("rail_march.xlsx", "WeightKG"), file_sum("road_march.csv", "WeightKG")
per_file_total = mar_w + rail_t * 1000 + road_w
v3 = per_file_total == int(fact["WeightKG"].sum())
print(f"3. Week 2 — marine {mar_w:,.0f} (text skipped) / rail {rail_t:,.2f} TONNES / road {road_w:,.0f}")
print(f"   converted sum {per_file_total:,.0f} vs warehouse {int(fact['WeightKG'].sum()):,} -> {'TRUE' if v3 else 'FALSE'}")
print("   different units plus text SUM skips — per-file SUMs are not comparable and do not add to 17,542,050")

# Claim 4 (Week 2): "The extract stage should drop the TOTAL footer from the road file, because it is obviously not a shipment."
road_rows = int((stg["SourceFile"] == "road_march.csv").sum())
road_arts = [a for a in artifacts if a[0] == "road_march.csv"]
footer_is_row = "TOTAL" in {r["ShipmentID"].strip().upper() for r in
                            stg[stg["SourceFile"] == "road_march.csv"].to_dict("records")}
v4 = footer_is_row  # the claim is true only if the footer would otherwise load as a row; it never does
print(f"4. Week 2 — road data rows: {road_rows}; road artifacts logged at extract: {road_arts} -> {'TRUE' if v4 else 'FALSE'}")
print("   settling number: 708 — extract corrects nothing; the TOTAL footer is a file artifact logged at extract, not a row the extract may delete")

# Claim 5 (Week 4): "The grain of FactShipment_March is one row per consignment."
v5 = len(fact) == fact["ShipmentID"].nunique()
print(f"5. Week 4 — {len(fact)} rows / {fact['ShipmentID'].nunique()} distinct ShipmentIDs -> {'TRUE' if v5 else 'FALSE'}")
print("   settling number: 1200 rows / 1200 distinct ShipmentIDs — after dedup on TRIM+UPPER ID, one row per consignment")

1. Week 5 — staging 1284 rows vs fact 1200 rows -> TRUE
   settling number: 1284 vs 1200 — staging holds every extracted row; only 1200 load
   (24 rejects + 60 duplicates = 84 = the 84-row gap)
2. Week 5 — negatives: 5 weights + 3 costs = 8 in total -> FALSE
   settling number: 3 — the 5 negative weights are errors, but the 3 negative costs are real credit notes, loaded as negatives
3. Week 2 — marine 4,683,000 (text skipped) / rail 9,218.25 TONNES / road 4,487,200
   converted sum 18,388,450 vs warehouse 17,542,050 -> FALSE
   different units plus text SUM skips — per-file SUMs are not comparable and do not add to 17,542,050
4. Week 2 — road data rows: 708; road artifacts logged at extract: [('road_march.csv', 710, 'blank line'), ('road_march.csv', 711, 'TOTAL footer')] -> FALSE
   settling number: 708 — extract corrects nothing; the TOTAL footer is a file artifact logged at extract, not a row the extract may delete
5. Week 4 — 1200 rows / 1200 distinct ShipmentIDs -> TRUE
   settlin

In [6]:
# Claims 6–10.
# Claim 6 (Week 1): "Average cost per kg across March can be computed by averaging the per-shipment cost per kg."
weighted = int(fact["FreightCostPKR"].sum()) / int(fact["WeightKG"].sum())
mean_ratios = (fact["FreightCostPKR"] / fact["WeightKG"]).mean()
v6 = abs(weighted - mean_ratios) < 1e-9
print(f"6. Week 1 — weighted {weighted:.4f} vs mean-of-ratios {mean_ratios:.4f} -> {'TRUE' if v6 else 'FALSE'}")
print("   settling number: weighted 8.3820 vs mean-of-ratios 7.7456 — cost per kg is a ratio of sums, not an average of ratios")

# Claim 7 (Week 3): "The right fix for the four border crossings is to add them to DimCity."
unk_o = loadable[loadable["Origin"].map(lambda v: alias.get((v or "").strip()) is None)]
spell_o = unk_o["Origin"].map(lambda v: (v or "").strip()).value_counts()
city_names = set(dim_city["CityName"])
added = [s for s in spell_o.index if s in city_names]
v7 = len(added) > 0
print(f"7. Week 3 — {len(unk_o)} rows / {len(spell_o)} spellings: {dict(spell_o)} -> {'TRUE' if v7 else 'FALSE'}")
print("   the 4 border-crossing spellings (Wagah Border 2, Taftan Crossing 2, Torkham Gate 2, Chaman Border 2) plus")
print("   Keti Bandar Jetty 3 and Kotri Junction 3 all route to Unknown -1 and are counted; inventing dimension")
print("   members mid-load would break T7 check 7")

# Claim 8 (Week 4): "The DimDate shipped in dims/ can serve the March fact table."
shipped_dd = pd.read_csv(DATA / "dims" / "DimDate.csv", dtype=str, keep_default_na=False)
march_keys = int((shipped_dd["MonthNumber"] == "3").sum())
v8 = march_keys >= 90
print(f"8. Week 4 — shipped DimDate: {len(shipped_dd)} rows, {shipped_dd['FullDate'].min()}..{shipped_dd['FullDate'].max()}, March keys {march_keys} (need 90) -> {'TRUE' if v8 else 'FALSE'}")
print("   settling number: 28 rows, all Feb 2026, 0 March keys — the shipped DimDate is February (stale)")

# Claim 9 (Lecture 11): "Snowflaking DimCity into city -> province -> country would make 'total weight by province' faster."
star_probes, snow_probes = 1 * len(fact), 3 * len(fact)
v9 = snow_probes < star_probes
print(f"9. Lecture 11 — snowflake {snow_probes} vs star {star_probes} probes -> {'TRUE' if v9 else 'FALSE'}")
print("   settling number: 3600 vs 1200 probes — snowflake traverses 3 joins per fact row; star reads Province straight off DimCity (1 join)")

# Claim 10 (Week 4): "A relationship from FactShipment_March to DimCity should be many-to-one, single direction."
n_origin_keys = fact["OriginCityKey"].nunique()
v10 = len(fact) == 1200 and n_origin_keys == 9
print(f"10. Week 4 — {len(fact)} fact rows -> {n_origin_keys} origin keys -> {'TRUE' if v10 else 'FALSE'}")
print("    settling number: 1200 fact rows -> 9 origin keys — many shipments per city, single direction")

6. Week 1 — weighted 8.3820 vs mean-of-ratios 7.7456 -> FALSE
   settling number: weighted 8.3820 vs mean-of-ratios 7.7456 — cost per kg is a ratio of sums, not an average of ratios
7. Week 3 — 14 rows / 6 spellings: {'Keti Bandar Jetty': 3, 'Kotri Junction': 3, 'Wagah Border': 2, 'Taftan Crossing': 2, 'Torkham Gate': 2, 'Chaman Border': 2} -> FALSE
   the 4 border-crossing spellings (Wagah Border 2, Taftan Crossing 2, Torkham Gate 2, Chaman Border 2) plus
   Keti Bandar Jetty 3 and Kotri Junction 3 all route to Unknown -1 and are counted; inventing dimension
   members mid-load would break T7 check 7
8. Week 4 — shipped DimDate: 28 rows, 2026-02-01..2026-02-28, March keys 0 (need 90) -> FALSE
   settling number: 28 rows, all Feb 2026, 0 March keys — the shipped DimDate is February (stale)
9. Lecture 11 — snowflake 3600 vs star 1200 probes -> FALSE
   settling number: 3600 vs 1200 probes — snowflake traverses 3 joins per fact row; star reads Province straight off DimCity (1 join)
10. W

In [7]:
# Claims 11–15.
# Claim 11 (Week 5): "Trim before deduplicate is a preference, not a constraint."
v11 = byte_removed == removed
print(f"11. Week 5 — dedup before trim removes {byte_removed}, after trim {removed} -> {'TRUE' if v11 else 'FALSE'}")
print(f"    settling number: {removed - byte_removed} — 12 trailing-space copies are invisible until trimmed")

# Claim 12 (Week 6): "If SUM(WeightKG) and SUM(FreightCostPKR) are both exact, the load is correct."
variant_rows = len(stg) - 15 - removed  # rejected only 15 rows, loaded the 9 text-weight rows
v12 = variant_rows == len(fact)
print(f"12. Week 6 — {len(stg)} - 15 - {removed} = {variant_rows} rows with exact totals -> {'TRUE' if v12 else 'FALSE'}")
print("    settling number: 1209 rows with exact totals — the 9 text-weight rows carry weight NULL/text, so SUM skips them;")
print("    row count and reject count expose it, the totals do not")

# Claim 13 (Lecture 12): "The six unparseable dates and the fourteen unresolved cities fail in the same way."
v13 = "REJECT" == "ROUTE"
print(f"13. Lecture 12 — {int(baddate.sum())} unparseable dates vs {len(unk_o)} unresolved cities -> {'TRUE' if v13 else 'FALSE'}")
print("    settling number: 6 vs 14 — unparseable dates cannot load at all (no DateKey) so REJECT; unresolved cities load with key -1 so ROUTE")

# Claim 14 (Lecture 11): "An index on ModeKey would do the most for the mode slicer, because that is what the slicer filters on."
mode_sel = fact["ModeKey"].nunique() / len(fact)
mode_worst = int(fact["ModeKey"].value_counts().max())
date_sel = fact["DateKey"].nunique() / len(fact)
v14 = mode_sel >= date_sel
print(f"14. Lecture 11 — ModeKey selectivity {mode_sel:.4f} ({fact['ModeKey'].nunique()} distinct; worst value {mode_worst} rows) vs DateKey {date_sel:.4f} -> {'TRUE' if v14 else 'FALSE'}")
print("    settling number: 0.0025 (3 distinct; worst value 660 rows) — a lookup still reads a third of the table; DateKey (0.0258) wins")

# Claim 15 (Lecture 12): "Deduplicating on day + lane + weight + customer is safe, because no two different consignments would match on all four."
loadable["_d"] = loadable["ShipDate"].map(parse_date)
loadable["_o"] = loadable["Origin"].map(lambda v: (v or "").strip())
loadable["_t"] = loadable["Destination"].map(lambda v: (v or "").strip())
loadable["_wkg"] = loadable.apply(lambda r: plain_number(r["WeightKG"]) * (1000 if "rail" in r["SourceFile"].lower() else 1), axis=1)
grp = loadable.groupby(["_d", "_o", "_t", "_wkg", "CustomerID"])["_n"].nunique()
pairs = grp[grp > 1]
for key in pairs.index:
    sub = loadable[(loadable["_d"] == key[0]) & (loadable["_o"] == key[1]) & (loadable["_t"] == key[2]) & (loadable["_wkg"] == key[3]) & (loadable["CustomerID"] == key[4])]
    print(f"    destroyed pair: {sorted(sub['_n'].unique())} — {key[0]} {key[1]}->{key[2]} {key[3]:,.0f} kg {key[4]}")
v15 = len(pairs) == 0
print(f"15. Lecture 12 — {len(pairs)} pairs: rd-5216/rd-5337; rd-5409/rd-5332 -> {'TRUE' if v15 else 'FALSE'}")
print("    two genuinely different consignments share day+lane+weight+customer — D4 merges them and bills one")

11. Week 5 — dedup before trim removes 48, after trim 60 -> FALSE
    settling number: 12 — 12 trailing-space copies are invisible until trimmed
12. Week 6 — 1284 - 15 - 60 = 1209 rows with exact totals -> FALSE
    settling number: 1209 rows with exact totals — the 9 text-weight rows carry weight NULL/text, so SUM skips them;
    row count and reject count expose it, the totals do not
13. Lecture 12 — 6 unparseable dates vs 14 unresolved cities -> FALSE
    settling number: 6 vs 14 — unparseable dates cannot load at all (no DateKey) so REJECT; unresolved cities load with key -1 so ROUTE
14. Lecture 11 — ModeKey selectivity 0.0025 (3 distinct; worst value 660 rows) vs DateKey 0.0258 -> FALSE
    settling number: 0.0025 (3 distinct; worst value 660 rows) — a lookup still reads a third of the table; DateKey (0.0258) wins


    destroyed pair: ['RD-5216', 'RD-5337'] — 2026-03-15 Lahore->Sialkot 6,300 kg CU-136
    destroyed pair: ['RD-5332', 'RD-5409'] — 2026-03-29 Karachi->Hyderabad 6,500 kg CU-102
15. Lecture 12 — 2 pairs: rd-5216/rd-5337; rd-5409/rd-5332 -> FALSE
    two genuinely different consignments share day+lane+weight+customer — D4 merges them and bills one


In [8]:
verdicts = [v1, v2, v3, v4, v5, v6, v7, v8, v9, v10, v11, v12, v13, v14, v15]
expected = [True, False, False, False, True, False, False, False, False, True, False, False, False, False, False]
assert verdicts == expected, f"verdicts disagree with the workbook: {verdicts}"
assert sum(verdicts) == 3, "self-check: exactly 3 of the fifteen are true"
print("SELF-CHECK PASS — exactly 3 true: #1, #5, #10")

# Analysis
# The pattern in the twelve falses: each one sounds like good practice
# (trust the file totals, drop the obvious footer, add the missing city,
# snowflake for speed, average the ratios) and each one breaks on a count
# you can only get by reading the files. That is why the sheet asks for the
# number, not the opinion.

SELF-CHECK PASS — exactly 3 true: #1, #5, #10
